In [1]:
# CELL 1: import libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import shap
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# CELL 2: load the dataset

DATA_PATH = "dataset location"  
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.head())
print(df.dtypes)

In [ ]:
# CELL 3: quick data analysis / target check

TARGET_COL = "target column"

# basic info
print("\nMissing values per column:")
print(df.isna().sum())

print("\nTarget distribution:")
print(df[TARGET_COL].value_counts(dropna=False))

# visualize target imbalance
plt.figure(figsize=(5,3))
df[TARGET_COL].value_counts().plot(kind="bar", color="teal")
plt.title("Target distribution (raw)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# CELL 4: separate features and target, detect numeric/categorical
# drop rows where target is missing, to keep things clean

df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

In [ ]:
# CELL 5: handle missing values
# numeric -> median

for col in numeric_cols:
    X[col] = X[col].fillna(X[col].median())

# categorical -> mode
for col in categorical_cols:
    if X[col].isna().any():
        X[col] = X[col].fillna(X[col].mode()[0])

print("Missing after imputation:")
print(X.isna().sum())

In [ ]:
# CELL 6: handle outliers (simple IQR capping on numeric columns)

def iqr_cap(series, factor=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 + ( -factor * iqr)
    upper = q3 + ( factor * iqr)
    return series.clip(lower, upper)

for col in numeric_cols:
    X[col] = iqr_cap(X[col])

print("Outlier capping done on numeric cols.")

In [ ]:
# CELL 7: encode categoricals and target
# encode feature categoricals with LabelEncoder (simple, works for small cardinality)

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

# encode target
target_le = LabelEncoder()
y_enc = target_le.fit_transform(y.astype(str))
class_names = target_le.classes_
print("Classes:", class_names)

In [ ]:
# CELL 8: train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.1, random_state=42, stratify=y_enc
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

In [ ]:
# CELL 9: scale numeric features (fit on train, apply on both)

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Scaling done.")

In [ ]:
# CELL 10: SMOTE on TRAIN ONLY and show augmented part

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", np.bincount(y_train))
print("After SMOTE:", np.bincount(y_train_sm))

# synthetic preview
orig_n = X_train_scaled.shape[0]
aug_X = X_train_sm.iloc[orig_n:].copy()
aug_y = y_train_sm[orig_n:]

aug_preview = aug_X.copy()
aug_preview[TARGET_COL] = target_le.inverse_transform(aug_y)
print("\nSynthetic samples preview:")
print(aug_preview.head(10))

In [ ]:
# CELL 11: reshape for 1D CNN (UPDATED for your 0_* and 1_* feature blocks)

import numpy as np

# The exact feature-type order (base names) for EACH block
base_feats = [
    "pre-RR", "post-RR", "pPeak", "tPeak", "rPeak", "sPeak", "qPeak",
    "qrs_interval", "pq_interval", "qt_interval", "st_interval",
    "qrs_morph0", "qrs_morph1", "qrs_morph2", "qrs_morph3", "qrs_morph4"
]

# Build ordered column lists
cols0 = [f"0_{b}" for b in base_feats]
cols1 = [f"1_{b}" for b in base_feats]
ordered_cols = cols0 + cols1

# Fail fast if something is missing (typo/mismatch in your dataframe)
missing_train = [c for c in ordered_cols if c not in X_train_sm.columns]
missing_test  = [c for c in ordered_cols if c not in X_test_scaled.columns]
if missing_train:
    raise ValueError(f"Missing columns in X_train_sm: {missing_train}")
if missing_test:
    raise ValueError(f"Missing columns in X_test_scaled: {missing_test}")

# Order columns consistently for train/test
X_train_ord = X_train_sm[ordered_cols]
X_test_ord  = X_test_scaled[ordered_cols]

# Reshape:
# steps = 16 feature types
# channels = 2 blocks (0 and 1)
X_train_cnn = X_train_ord.to_numpy().reshape(-1, len(base_feats), 2).astype(np.float32)
X_test_bal_cnn = X_test_ord.to_numpy().reshape(-1, len(base_feats), 2).astype(np.float32)

num_classes = len(np.unique(y_train_sm))

print("CNN input shape:", X_train_cnn.shape)      # (N, 16, 2)
print("Test input shape:", X_test_bal_cnn.shape)  # (N, 16, 2)
print("Num classes:", num_classes)

In [13]:
# CELL 12: channel attention (squeeze–excitation style)
# CELL 12: channel attention (UNCHANGED)
from tensorflow.keras import layers

def channel_attention_1d(inputs, reduction=8):
    ch = int(inputs.shape[-1])
    gap = layers.GlobalAveragePooling1D()(inputs)
    dense1 = layers.Dense(max(ch // reduction, 1), activation="relu")(gap)
    dense2 = layers.Dense(ch, activation="sigmoid")(dense1)
    scale = layers.Multiply()([inputs, layers.Reshape((1, ch))(dense2)])
    return scale


In [29]:
# CELL 13: build CNN + Channel Attention model (UPDATED input_shape)
from tensorflow import keras

def build_cam_cnn(input_shape, num_classes):
    inp = keras.Input(shape=input_shape)

    x = layers.Conv1D(64, 3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = channel_attention_1d(x)

    x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = channel_attention_1d(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Input is now (steps=16, channels=2)
model = build_cam_cnn((len(base_feats), 2), num_classes)
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 16, 2)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 16, 64)    │        448 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 64)    │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 8)         │        520 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │        576 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 64)     │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 16, 64)    │          0 │ batch_normalizat… │
│                     │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 16, 128)   │     24,704 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 128)   │        512 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 16)        │      2,064 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 128)       │      2,176 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 128)    │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 16, 128)   │          0 │ batch_normalizat… │
│ (Multiply)          │                   │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ multiply_1[0][0]  │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │     16,512 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 4)         │        516 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 48,284 (188.61 KB)

 Trainable params: 47,900 (187.11 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
# CELL 14: train the model

EPOCHS = 20
BATCH_SIZE = 64

history = model.fit(
    X_train_cnn, y_train_sm,
    validation_split=0.15,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
# CELL 15: evaluate on the test set

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_prob = model.predict(X_test_bal_cnn)
y_pred = np.argmax(y_prob, axis=1)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=class_names))

cm = confusion_matrix(y_test_bal, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (INCART 2-lead Arrhythmia Database)")
plt.tight_layout()
plt.show()

In [ ]:
# CELL 15B: Plot ROC Curves (per-class only, no micro/macro)

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from itertools import cycle

# One-vs-rest ROC setup
y_test_bin = label_binarize(y_test_bal, classes=np.arange(num_classes))

plt.figure(figsize=(7, 6))
colors = cycle(["red", "blue", "green", "purple", "orange", "teal", "brown"])

for i, color in zip(range(num_classes), colors):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(
        fpr, tpr,
        color=color,
        lw=2,
        label=f"{class_names[i]} (AUC = {roc_auc:.2f})"
    )

plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (INCART 2-lead Arrhythmia Database)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()